# Analyse de paniers : Apriori, FP-Growth et recommandations

Ce notebook suit les différentes étapes du mini-projet. Les résultats chiffrés sont calculés à l'exécution : ils ne sont pas préremplis. Les deux fichiers du dossier `data/` sont des **CSV** (leur nom contient aussi `.xlsx`, mais leur contenu est bien du texte CSV). Le fichier réel contient environ 542 000 lignes ; pour maîtriser le coût mémoire et rendre la comparaison Apriori/FP-Growth praticable, l'extraction principale est faite sur le Royaume-Uni, qui constitue le plus grand marché. Les statistiques descriptives portent, elles, sur toutes les données nettoyées.

Installer les dépendances dans le kernel si nécessaire : `%pip install -r requirements.txt` depuis la racine du projet. Les algorithmes limitent les itemsets à 3 articles pour maîtriser l'explosion combinatoire ; le même réglage est utilisé pour les deux méthodes comparées.

In [ ]:
from pathlib import Path
from time import perf_counter
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for package in ("apyori", "mlxtend"):
    if importlib.util.find_spec(package) is None:
        raise ImportError(f"Dépendance absente : {package}. Exécutez %pip install -r requirements.txt puis redémarrez le kernel.")

from apyori import apriori as apyori_apriori
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
ORDER_FILE = DATA_DIR / "order_data.xlsx - order_data.csv"
RETAIL_FILE = DATA_DIR / "transaction_dataset.xlsx - transaction_dataset.csv"
assert ORDER_FILE.is_file(), f"Fichier introuvable : {ORDER_FILE}"
assert RETAIL_FILE.is_file(), f"Fichier introuvable : {RETAIL_FILE}"
print(f"Racine du projet : {ROOT}")

## Partie 0 — Prise en main et Apriori (`apyori`)

Le petit fichier est un texte sans en-tête : chaque ligne correspond à un panier et les produits sont séparés par des espaces. Les seuils demandés sont appliqués strictement.

In [ ]:
order_df = pd.read_csv(ORDER_FILE, sep=r"\s+", header=None, dtype=str, engine="python")
small_transactions = [row.dropna().tolist() for _, row in order_df.iterrows()]
small_transactions = [basket for basket in small_transactions if basket]
print(f"{len(small_transactions)} transactions ; {sum(map(len, small_transactions))} occurrences de produits")
small_transactions[:3]

In [ ]:
small_result = list(apyori_apriori(
    small_transactions,
    min_support=0.25,
    min_confidence=0.2,
    min_lift=2,
    min_length=2,
))

small_rules = []
for itemset_result in small_result:
    for statistic in itemset_result.ordered_statistics:
        if statistic.items_base and statistic.items_add:
            small_rules.append({
                "antecedents": tuple(sorted(statistic.items_base)),
                "consequents": tuple(sorted(statistic.items_add)),
                "support": itemset_result.support,
                "confidence": statistic.confidence,
                "lift": statistic.lift,
            })
small_rules_df = (pd.DataFrame(small_rules)
                  .drop_duplicates(subset=["antecedents", "consequents"])
                  .sort_values(["lift", "confidence"], ascending=False)
                  .head(5))
if small_rules_df.empty:
    print("Aucune règle ne satisfait simultanément les quatre seuils sur ce petit jeu de données.")
else:
    display(small_rules_df)
    for _, rule in small_rules_df.head(2).iterrows():
        left = " + ".join(rule["antecedents"])
        right = " + ".join(rule["consequents"])
        print(f"\n$\\{{{left}\\}} \\Rightarrow \\{{{right}\\}}$")
        print(f"Métier : {rule['support']:.1%} des paniers contiennent les deux côtés ; parmi les clients qui achètent {left}, {rule['confidence']:.1%} achètent aussi {right}. Le lift de {rule['lift']:.2f} signifie que cette co-achat est {rule['lift']:.2f} fois plus fréquent que si les achats étaient indépendants.")

## Partie 1 — Préparation et statistiques descriptives

On conserve les identifiants de facture et de produit, la description, la quantité, le pays et la date. Les factures d'annulation (identifiant commençant par `C`), les quantités nulles/négatives, les dates invalides et les lignes incomplètes sont écartées. Une facture est ensuite représentée par l'ensemble de ses descriptions de produits : acheter plusieurs unités du même article ne crée pas plusieurs items dans un panier.

In [ ]:
columns = ["InvoiceNo", "StockCode", "Description", "Quantity", "Country", "InvoiceDate"]
retail = pd.read_csv(
    RETAIL_FILE,
    usecols=columns,
    dtype={"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "string"},
    low_memory=False,
)
retail["Quantity"] = pd.to_numeric(retail["Quantity"], errors="coerce")
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], format="%m/%d/%Y %H:%M", errors="coerce")
retail = retail.dropna(subset=columns).copy()
retail["Description"] = retail["Description"].str.strip().str.upper()
retail = retail.loc[
    (retail["Quantity"] > 0)
    & ~retail["InvoiceNo"].str.startswith("C", na=False)
    & retail["Description"].ne("")
].copy()

all_baskets = retail.groupby("InvoiceNo", sort=False)["Description"].agg(
    lambda products: sorted(set(products))
).tolist()
transaction_count = len(all_baskets)
distinct_products = retail["Description"].nunique()
print(f"Transactions nettoyées : {transaction_count:,}")
print(f"Articles distincts vendus : {distinct_products:,}")
print("Trois exemples de paniers :")
for basket in all_baskets[:3]:
    print("-", basket[:12], "..." if len(basket) > 12 else "")

## Partie 2 — FP-Growth, règles et actions marketing

Le Royaume-Uni est utilisé pour l'extraction principale afin de comparer les algorithmes sur un volume représentatif sans transformer l'encodage en matrice démesurée. Le seuil de support de 3 % élimine les co-achats trop rares pour une action commerciale stable ; la longueur maximale de 3 limite la combinatoire. Les colonnes de la matrice indiquent la présence (`True`/`False`) d'une description dans une facture.

In [ ]:
MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.5
MAX_ITEMSET_LENGTH = 3
MAIN_COUNTRY = "United Kingdom"

uk_invoice_ids = set(retail.loc[retail["Country"].eq(MAIN_COUNTRY), "InvoiceNo"].unique())
uk_baskets = retail.loc[retail["InvoiceNo"].isin(uk_invoice_ids)].groupby(
    "InvoiceNo", sort=False
)["Description"].agg(lambda products: sorted(set(products))).tolist()
print(f"Paniers UK : {len(uk_baskets):,}")

encoder = TransactionEncoder()
encoded = encoder.fit(uk_baskets).transform(uk_baskets)
basket_matrix = pd.DataFrame(encoded, columns=encoder.columns_, dtype=bool)
print(f"Matrice booléenne : {basket_matrix.shape[0]:,} factures x {basket_matrix.shape[1]:,} produits")

In [ ]:
frequent_itemsets = fpgrowth(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LENGTH,
)
top_itemsets = frequent_itemsets.sort_values("support", ascending=False).head(10).copy()
top_itemsets["itemsets"] = top_itemsets["itemsets"].apply(lambda items: " + ".join(sorted(items)))
display(top_itemsets)

if frequent_itemsets.empty:
    rules = pd.DataFrame()
    print("Aucun itemset fréquent : essayez de diminuer MIN_SUPPORT.")
else:
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=MIN_CONFIDENCE,
    ).sort_values(["lift", "confidence", "support"], ascending=False)
    display(rules.head(10))
    if rules.empty:
        print("Aucune règle ne dépasse le seuil de confiance ; réduisez MIN_CONFIDENCE pour explorer.")

In [ ]:
if not rules.empty:
    selected_rules = rules.loc[(rules["lift"] > 1.2) & (rules["support"] >= MIN_SUPPORT)].head(5)
    if selected_rules.empty:
        print("Aucune règle ne dépasse lift > 1.2 ; voici les règles au lift le plus élevé comme pistes exploratoires.")
        selected_rules = rules.head(5)
    for number, (_, rule) in enumerate(selected_rules.iterrows(), start=1):
        antecedents = sorted(rule["antecedents"])
        consequents = sorted(rule["consequents"])
        left = " et ".join(antecedents)
        right = " et ".join(consequents)
        action = ("proposer un bundle avec un test de remise modérée" if rule["support"] >= 0.05
                  else "tester un pop-up « Vous aimerez aussi… » ciblé, sans imposer le bundle")
        print(f"{number}. Les clients qui achètent {left} achètent souvent {right}.")
        print(f"   Support = {rule['support']:.3%} ; confiance = {rule['confidence']:.3%} ; lift = {rule['lift']:.2f}.")
        print(f"   Action proposée : {action} ; mesurer la marge avant d'étendre l'opération.\n")
else:
    print("Pas de règle à traduire avec les seuils choisis.")

## Partie 3 — Comparaison algorithmique

Les deux implémentations travaillent sur la même matrice, avec le même support et la même longueur maximale. `perf_counter` remplace la magie Jupyter `%time` afin que le code fonctionne aussi lors d'une exécution comme script. Le temps dépend du matériel et de la charge ; la comparaison de règles est calculée sur le même seuil de confiance.

In [ ]:
start = perf_counter()
apriori_itemsets = apriori(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LENGTH,
)
apriori_seconds = perf_counter() - start

start = perf_counter()
fp_itemsets_benchmark = fpgrowth(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LENGTH,
)
fpgrowth_seconds = perf_counter() - start

def rules_from(itemsets):
    if itemsets.empty:
        return pd.DataFrame()
    return association_rules(itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)

apriori_rules = rules_from(apriori_itemsets)
fp_rules = rules_from(fp_itemsets_benchmark)
comparison = pd.DataFrame([
    {"Algorithme": "Apriori", "Secondes": apriori_seconds, "Itemsets fréquents": len(apriori_itemsets), "Règles": len(apriori_rules)},
    {"Algorithme": "FP-Growth", "Secondes": fpgrowth_seconds, "Itemsets fréquents": len(fp_itemsets_benchmark), "Règles": len(fp_rules)},
])
display(comparison)
for name, frame in (("Apriori", apriori_rules), ("FP-Growth", fp_rules)):
    unique_count = (len(frame.drop_duplicates(subset=["antecedents", "consequents"])) if not frame.empty else 0)
    print(f"{name} : {len(frame) - unique_count} doublon(s) de règles au sens des côtés identiques.")

if apriori_seconds > 0 and fpgrowth_seconds > 0:
    faster = "FP-Growth" if fpgrowth_seconds < apriori_seconds else "Apriori"
    print(f"Sur cette exécution, {faster} est le plus rapide. Pour de grands volumes, FP-Growth est généralement mieux adapté car il évite la génération explicite de nombreux candidats ; la mesure locale reste à interpréter avec le nombre d'items et le matériel.")

## Extensions — sensibilité, segmentation et recul critique

### Sensibilité au support

Une hausse du support élimine les co-achats rares : les règles très communes peuvent être peu différenciantes, tandis que les règles rares mais solides demandent assez de transactions pour être fiables. Les résultats ci-dessous mesurent l'effet des seuils demandés sur le même marché UK.

In [ ]:
support_values = [0.01, 0.02, 0.03, 0.05]
sensitivity_rows = []
for support in support_values:
    itemsets = fpgrowth(
        basket_matrix,
        min_support=support,
        use_colnames=True,
        max_len=MAX_ITEMSET_LENGTH,
    )
    current_rules = rules_from(itemsets)
    sensitivity_rows.append({"Support minimum": support, "Itemsets": len(itemsets), "Règles (confiance >= 0.5)": len(current_rules)})
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
ax = sensitivity.plot(x="Support minimum", y=["Itemsets", "Règles (confiance >= 0.5)"], marker="o", figsize=(8, 4))
ax.set(title="Nombre d'itemsets et de règles selon le support", ylabel="Nombre", xlabel="Support minimum")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
international = retail.loc[retail["Country"].ne(MAIN_COUNTRY)]
international_baskets = international.groupby("InvoiceNo", sort=False)["Description"].agg(
    lambda products: sorted(set(products))
).tolist()
if len(international_baskets) < 2:
    print("Pas assez de factures internationales pour comparer les marchés.")
else:
    international_encoder = TransactionEncoder()
    international_matrix = pd.DataFrame(
        international_encoder.fit(international_baskets).transform(international_baskets),
        columns=international_encoder.columns_,
        dtype=bool,
    )
    international_itemsets = fpgrowth(
        international_matrix,
        min_support=MIN_SUPPORT,
        use_colnames=True,
        max_len=MAX_ITEMSET_LENGTH,
    )
    international_rules = rules_from(international_itemsets)
    def rule_signatures(frame):
        if frame.empty:
            return set()
        return {(frozenset(a), frozenset(c)) for a, c in zip(frame["antecedents"], frame["consequents"])}
    uk_signatures = rule_signatures(rules)
    international_signatures = rule_signatures(international_rules)
    print(f"Factures internationales : {len(international_baskets):,}")
    print(f"Règles communes aux deux marchés : {len(uk_signatures & international_signatures)}")
    print(f"Règles spécifiques UK : {len(uk_signatures - international_signatures)}")
    print(f"Règles spécifiques international : {len(international_signatures - uk_signatures)}")
    print("Les règles communes aux marchés (mêmes supports minimums) sont des candidates universelles ; afficher les règles propres à un marché pour guider une campagne locale.")

### Conclusion et pistes métier

- **Choix algorithmique :** FP-Growth est généralement adapté aux grands volumes, car il compresse les transactions et évite d'énumérer les candidats comme Apriori. La mesure locale ci-dessus permet de vérifier ce constat sur cet échantillon et ce matériel.
- **Limites :** une association mesure une co-occurrence, pas une causalité ; les règles varient selon la période, le pays, la disponibilité, les promotions et la popularité des produits. Les descriptions proches ou articles génériques peuvent dominer les résultats, et les associations répétitives réduisent la lisibilité.
- **Améliorations :** (1) recalculer les règles par trimestre ou saison et comparer leur stabilité ; (2) exclure les articles très génériques ou analyser des catégories/SKU normalisés avant extraction.

### Transformer les règles en recommandations

Le module « Fréquemment achetés ensemble » peut sélectionner une règle dont l'antécédent est présent dans le panier, puis proposer un article du conséquent, en filtrant les produits indisponibles et en variant les recommandations pour préserver la diversité. Surveiller le taux de clic, le taux d'ajout au panier, la conversion, le panier moyen, la marge, les retours et la diversité des produits recommandés. Pour valider l'effet, répartir aléatoirement les visiteurs éligibles entre une expérience avec recommandations et un groupe témoin, définir une durée et une métrique principale à l'avance, puis comparer les résultats avec incertitude statistique et garde-fous de marge/retours.

Ne pas interpréter « A est acheté avec B » comme « A provoque l'achat de B ». La saisonnalité et les promotions peuvent créer ou masquer des associations ; une personnalisation trop répétitive risque d'enfermer les clients dans une bulle et de réduire la découverte de nouveaux produits.